## Levantar el entorno

Descarga la data (si falta) y levanta el cluster de Spark (master + 2 workers) y Jupyter con Docker

- Kernel picker → "Existing Jupyter Server..." → `http://localhost:8888`
- UI de Spark: `http://localhost:8080` (master), `http://localhost:4040` (job activo)

In [1]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent
print("Proyecto:", proyecto)

Proyecto: c:\Users\rexFe\OneDrive\Documentos\DataMining\Proyecto2\dm-proj2


### 1. Docker disponible

In [2]:
r = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
print(r.stdout or "Docker no responde: abre Docker Desktop y vuelve a ejecutar")

Docker Compose version v5.6.0



### 2. Descarga de la data (OECE, 2023-2025)

Se omite si `data/raw` ya tiene los JSON

In [3]:
raw = proyecto / "data" / "raw"
if not any(raw.glob("*/*.json")):
    subprocess.run([sys.executable, str(proyecto / "data" / "download.py")], check=True)
print(f"{len(list(raw.glob('*/*.json')))} archivos en data/raw")

36 archivos en data/raw


### 3. Levantar contenedores

In [4]:
env = {**os.environ}
if hasattr(os, "getuid"):
    env.setdefault("HOST_UID", str(os.getuid()))
r = subprocess.run(["docker", "compose", "up", "-d"], cwd=proyecto, env=env, capture_output=True, text=True)
print(r.stdout, r.stderr)

  Container dm-proj2-spark-master-1 Recreate 
 Container dm-proj2-spark-master-1 Recreated 
 Container dm-proj2-spark-worker-1 Recreate 
 Container dm-proj2-spark-worker-2-1 Recreate 
 Container dm-proj2-spark-worker-2-1 Recreated 
 Container dm-proj2-spark-worker-1 Recreated 
 Container dm-proj2-jupyter-1 Recreate 
 Container dm-proj2-jupyter-1 Recreated 
 Container dm-proj2-spark-master-1 Starting 
 Container dm-proj2-spark-master-1 Started 
 Container dm-proj2-spark-worker-2-1 Starting 
 Container dm-proj2-spark-worker-1 Starting 
 Container dm-proj2-spark-worker-2-1 Started 
 Container dm-proj2-spark-worker-1 Started 
 Container dm-proj2-jupyter-1 Starting 
 Container dm-proj2-jupyter-1 Started 



### 4. Esperar a que respondan los servicios

In [5]:
def esperar(url, intentos=60):
    for _ in range(intentos):
        try:
            urllib.request.urlopen(url, timeout=3)
            return True
        except Exception:
            time.sleep(5)
    return False

for nombre, url in {"Spark master": "http://localhost:8080", "Jupyter": "http://localhost:8888/api"}.items():
    print(f"{nombre:13s}: {'OK' if esperar(url) else 'NO RESPONDE'}  {url}")

Spark master : OK  http://localhost:8080
Jupyter      : OK  http://localhost:8888/api


In [6]:
estado = json.loads(urllib.request.urlopen("http://localhost:8080/json/").read())
print("workers vivos:", sum(w["state"] == "ALIVE" for w in estado["workers"]), "| cores:", estado["cores"])
print(subprocess.run(["docker", "compose", "ps"], cwd=proyecto, capture_output=True, text=True).stdout)

workers vivos: 2 | cores: 2
NAME                        IMAGE                                          COMMAND                  SERVICE          CREATED         STATUS                            PORTS
dm-proj2-jupyter-1          quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   jupyter          8 seconds ago   Up 6 seconds (health: starting)   0.0.0.0:4040->4040/tcp, [::]:4040->4040/tcp, 0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp
dm-proj2-spark-master-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-master     8 seconds ago   Up 7 seconds (health: starting)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
dm-proj2-spark-worker-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker     8 seconds ago   Up 6 seconds (health: starting)   0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp
dm-proj2-spark-worker-2-1   quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker-2   8 se